In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

TODO:

- Try class for `Card` later? `namedtuple` works for now.
- Try `IntEnum` for Poker hand rankings. Currently, we use integer values from 14 to 1 to indicate the poker hand rankings. Using `IntEnum` may make it more readable. Read more about enum: https://docs.python.org/3/library/enum.html
- Try passing `Round` to `Player` when taking action to update `stage_bet`.
- Player should not be able to check if `stage_bet != player.bet`.
- Add possible actions for players.

In [ ]:
from fastcore.utils import *
from enum import Enum, auto
from dataclasses import dataclass, field

# core

> Fill in a module description here

In this dialog, I want to go through the process of writing a poker program myself but following the approach I learned in the cs212 class. I went through the lesson in `pytude/cs212/lesson_1` dialog. Read the dialog to guide me when I ask for help without giving me answers directly. It's been some time since I went over this lesson, so I mostly likely forgot about all the details of implementations. My goal is not to copy everything from the lesson exactly, but follow the steps: understanding the problem, specifying, concept inventory, etc. that I can apply to solving any problem. To challenge myself, I will write texas hold'em poker.

Here's the process Norvig taught:

**1. Understand** — Start with a vague understanding and refine it. Make an inventory of all the concepts you'll need to deal with (data types, operations, etc.).

**2. Specify** — Define *what* needs to happen: inputs, outputs, what each function should accomplish. Think of it as a contract — not the implementation details, just the "what."

**3. Design** — Figure out *how* to make it happen — the actual algorithms, code structure, and implementation.

Along the way, key principles:
- **Write tests early** (even before implementation), including **extreme values**
- **Reuse existing tools** when possible
- **Refactor** for elegance (DRY, clarity, simplicity, generality)
- Be aware of the **four dimensions**: correctness, efficiency, features, elegance — and make conscious tradeoffs

## Understanding the problem.

Rules in the poker:

Poker is a card game with 52 cards, and each card has a suit and a rank. A suit consists of spades, clubs, hearts, and diamonds. A rank consists of numbers from ace, 2 to 10, jack, queen, and king. A hand has two cards, and community cards have five cards. They are combined to make different combinations used for comparisons among hands. At the start of the round, each player gets two cards face down and three cards in the community. After each betting round, one additional card is dealt to the community until there are five cards. Each hand from the players is compared, and winner takes all the bets unless there is a tie, and they split. To compare the hands, poker ranking is used. At each betting round, players can call, check, raise, or fold. If everyone folds except for one player, that player wins. If a player raises, others have to match the bet (call) or raise, or fold. After everyone checks, next round starts. 

Poker hands:
- royal flush
- straight flush
- four of a kind
- full house
- flush
- straight
- three of a kind
- two pair
- one pair
- high card

Concept inventory

Core concepts:
  - `Rank`: 2 to 10, jack, queen, king, and ace
  - `suit`: diamonds, hearts, clubs, spades.
  - `Card`: with rank and suit.
  - `CommunityCards`: Three to five cards shared by all the players.
  - `Hand`: 2 hole cards.
  - `HandRank`: Combining `Hand` with `CommunityCards`, hand rankings are compared among the players. From 7 cards, best 5 cards are used.
  - `Stage`: A stage is where each player takes actions such as call, fold, raise, and check. It ends when everyone folds except for those who called.
  - `Round`: A round consists of four stages (pre-flop, after flop, turn, and river). 
  - `Stage_bet`: money to match each round for each player and becomes the prize pool.
  - `Bet`: Money each player put in this stage.
  - `Balance`: The amount of moeny each player possesses so they can use for betting.
  - `Player state`: Useful player state such as balance, hand, whether playing the round or not, and bet.
  - `Deck`: 52 cards that get shuffled.
  - `Pot`: prize money that gets distributed to winners at the end of each round.

Add-ons:
  - `Blinds`: small and big blind. Small is immediately to the left of the dealer, and big blind is after. Big blind is twice the small blind amount.
  - `Dealer`: rotates clockwise each hand.
  - `display`: text based or graphical display.
  - `GameObserver/AI advisor`: sees the public game state and hole cards. Answers questions on request.

Operations

- Shuffle the deck
- Deal cards
- Perform player actions: call, fold, check, raise
- Evaluate a hand rank
- Compare hand ranks
- Distribute the pot to winners
- Start/end rounds
- Rotate dealer
- Draw community cards

**Specification**

Specifying is the step where you define **what** your program should do, without worrying about *how* it does it. It's like writing a contract:

- What are the **inputs**?
- What are the **outputs**?
- What **behavior** should the program have?

--- 

Card & Deck Operations
- `shuffle(deck) -> None`: Shuffle the deck.
- `deal_hole(deck, players) -> None`: Deal 2 cards to each player.
- `deal_community(deck, community_cards, n) -> None`: Deal n cards from deck to community cards.

Hand Evaluation
  - `evaluate_hand(hand, community_cards) -> hand_rank`: From 7 cards, find the best 5-card hand and return its rank.
  - `compare_ranks(hand_ranks) -> tuple`: Given a set of hand ranks, return the best one.

Player Actions
  - `perform_action(player, round) -> None`: Prompt player for an action, then call the appropriate function.
  - `call(player, round) -> None`: Player matches the current bet. Round state updated.
  - `fold(player, round) -> None`: Player folds. Round state updated.
  - `check(player, round) -> None`: Player checks. Round state updated.
  - `raise(player, round, amount) -> None`: Player raises by amount. Round state updated.

Round Management
  - `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
  - `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.
  - `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).

State
- round: tracks players with their bets, pot, community cards, and current betting state.
- player: has balance, current bet, hand, and whether folded or active.

**Design**

Design is where you figure out **how** to make it happen. The spec said *what* your program does; design says *how* it does it.

This is where you make decisions like:

- **Data representation** — How do you represent a card? A tuple `(rank, suit)`? A string `"AS"`? A custom class?
- **Code structure** — How do you organize the functions? What goes in a class vs standalone functions?
- **Algorithms** — How do you evaluate the best 5-card hand from 7? How do you determine when a betting round is over?

It's still not about writing the final code — it's about making the key implementation choices before you start coding.


`Card`:
- In the beginning, I tried using a string "♠2". This was a very simple and easy way to represent a card. To grab a rank or a suit from a card, `c1`, I can do `c1[0]` or `c1[1:]`. To make it more readable, I defined a function for these operations. Then I thought it was more convenient to use a dot notation. Rather than using a class, I tried using `namedtuple` from `collections`. This was a very light weight.

class `Player` with attributes 
- balance: int
- bet: int
- playing: bool
- hand: List[Card]
methods: 
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount). 

All the methods need to pass `round` and update it. Therefore, I might get a `round` attribute. But this leads to circular dependency, which tightly couples these together. It becomes harder to test.

`Round` class attributes are 
- players: List[Player]
- dealer: int
- deck: Deck
- stage: Enum(pre-flop, flop, turn, river)
- pot: int
- community: CommunityCard

methods:
- betting_round()
- start_round()
- distribute()

**Architecture**

`Game` is the top-level mediator. It owns `players` and the current `round`. Neither `Player` nor `Round` reference each other directly — `Game` coordinates between them.

**State ownership**

`Round` is the single owner of all mutable state: player balances, bets, pot, community cards, dealer position, and stage. All state updates flow through `Round`.

**Functional data operations**

`Deck` and `Card` operations are pure — they return new values rather than mutating:

- `shuffle() -> Deck` — returns a new shuffled deck
- `deal(n) -> (cards, Deck)` — returns dealt cards and a new deck with remaining cards

**Pure logic functions**

Hand evaluation and bet calculations:

- `evaluate_hand(hand, community) -> HandRank` — given 7 cards, returns the best 5-card rank
- `compare_ranks(ranks) -> tuple` — returns the winner rank

**Round methods**

`Round` applies results from pure functions to update state:

- `betting_round()` — manages one round of actions
- `start_round()` — orchestrates the full hand
- `distribute(winners)` — distributes pot to winners

**Player actions**

Players receive only the data they need (current bet, pot, etc.) rather than a full `Round` reference. Actions return *what* the new state should be, and `Round` applies it.

How to compare hands? Let's use a tuple with first entry representing the `hand_rank` from royal flush to hgih card with 10 to 1. Then later entries for tie breakers.

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

## Card

There are many ways to design `Card`. One of the simplest way is to use string.

In [ ]:
Card = str

I use emoji for suits and string of alphanumerics for ranks.

In [ ]:
suits = '♠♥♦♣'
suits

'♠♥♦♣'

In [ ]:
ranks = '2,3,4,5,6,7,8,9,10,J,Q,K,A'.split(',')
ranks

['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

In [ ]:
Card(suits[0] + ranks[-1])

'♠A'

Rather than using a string for cards and using separate functions to access suits and ranks, I thought it would be better to use the `.` (dot) notation. Rather than using a `class`, I wanted to try using `namedtuple`, which allows me to use attribute access and immutability. The ranks and suits from each card should not change. It is more lightweight than `class`.

In [ ]:
from collections import namedtuple
Card = namedtuple('Card', ['suit', 'rank'])
Card

__main__.Card

In [ ]:
c1 = Card('♠', '10')
c1.suit, c1.rank

('♠', '10')

In [ ]:
c1

Card(suit='♠', rank='10')

I want the card representation to be more concise, so I patch it.

In [ ]:
@patch
def __repr__(self: Card): return self.suit + self.rank

In [ ]:
c2 = Card('♠', 'K')
c2

♠K

Currently, we cannot compare the ranks directly as we are using string comparisons lexicographically. 

In [ ]:
c1.rank > c2.rank, '10' > 'K'

(False, False)

It is convenient to sort hands by rank values when evaluating hands. 

Using `rank_values` mapping, we can convert the string ranks into integers. From 2 to 10, their values are 2 to 10 respectively. Then, from 'J' to 'A', 11 to 14.

In [ ]:
rank_values = {r:i for i,r in enumerate(ranks, start=2)}
rank_values

{'2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '10': 10,
 'J': 11,
 'Q': 12,
 'K': 13,
 'A': 14}

Although I had `sort_cards` function that sorts cards by their `rank_values` from the highest, it was more convenient to patch `__lt__` and `__eq__` and just use default `sorted` function. When comparing cards, we only care about their ranks, not ranks. 

In [ ]:
@patch
def __lt__(self: Card, other: Card): return rank_values[self.rank] < rank_values[other.rank]

In [ ]:
@patch
def __eq__(self: Card, other: Card): return rank_values[self.rank] == rank_values[other.rank]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
sorted(deck,reverse=True)[:5]

[♠A, ♥A, ♦A, ♣A, ♠K]

## Hand

- `evaluate_hand(cards) -> HandRank` — given 7 cards, returns the best 5-card `HandRank`
- `compare_ranks(ranks) -> tuple` — returns the winner rank

In [ ]:
from collections import Counter

A hand has two cards. Community cards have five cards. 

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

In [ ]:
c1, c2

(♠10, ♠K)

In [ ]:
c1.rank

'10'

### Sample hands

These are some example hands.

In [ ]:
cards_highcard = [Card(suits[0], ranks[0]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4]),
    Card(suits[3], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[10]), Card(suits[2], ranks[12])]
cards_highcard

[♠2, ♥4, ♦6, ♣8, ♠10, ♥Q, ♦A]

In [ ]:
cards_2kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[3]), Card(suits[1], ranks[5]), 
    Card(suits[2], ranks[7]), Card(suits[3], ranks[9]), Card(suits[0], ranks[11])]
cards_2kind

[♠2, ♥2, ♠5, ♥7, ♦9, ♣J, ♠K]

In [ ]:
cards_3kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[0], ranks[3]), 
    Card(suits[1], ranks[5]), Card(suits[2], ranks[7]), Card(suits[3], ranks[9])]
cards_3kind

[♠2, ♥2, ♦2, ♠5, ♥7, ♦9, ♣J]

In [ ]:
cards_4kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[3], ranks[0]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[7]), Card(suits[2], ranks[10])]
cards_4kind

[♠2, ♥2, ♦2, ♣2, ♠6, ♥9, ♦Q]

In [ ]:
cards_2pair = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[1]), Card(suits[3], ranks[1]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_2pair

[♠2, ♥2, ♦3, ♣3, ♠6, ♥8, ♦10]

In [ ]:
cards_fullhouse = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[1]), Card(suits[2], ranks[1]), 
    Card(suits[3], ranks[1]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_fullhouse

[♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10]

In [ ]:
cards_straight = [Card(suits[0], ranks[1]), Card(suits[1], ranks[2]), Card(suits[2], ranks[3]),
    Card(suits[3], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straight

[♠3, ♥4, ♦5, ♣6, ♠7, ♥10, ♦Q]

In [ ]:
cards_flush = [Card(suits[0], ranks[0]), Card(suits[0], ranks[2]), Card(suits[0], ranks[4]),
    Card(suits[0], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[3]), Card(suits[2], ranks[7])]
cards_flush

[♠2, ♠4, ♠6, ♠8, ♠10, ♥5, ♦9]

In [ ]:
cards_straightflush = [Card(suits[0], ranks[1]), Card(suits[0], ranks[2]), Card(suits[0], ranks[3]),
    Card(suits[0], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straightflush

[♠3, ♠4, ♠5, ♠6, ♠7, ♥10, ♦Q]

In [ ]:
cards_royalflush = [Card(suits[0], ranks[12]), Card(suits[0], ranks[11]), Card(suits[0], ranks[10]), 
    Card(suits[0], ranks[9]), Card(suits[0], ranks[8]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4])]
cards_royalflush

[♠A, ♠K, ♠Q, ♠J, ♠10, ♥4, ♦6]

### Ranks and counts

By getting the counts from the cards, we can figure out `four of a kind`, `full house`, `three of a kind`, `two pair`, and `one pair`. One thing we have to be careful of is because we have seven cards to evaluate, `full house` might have two `three of kind`s, rather than a `three of kind` and a `pair`.

Here is `get_ranks_counts`, which takes a list of cards and returns a tuple of ranks and counts. The output is sorted by the rank counts and `rank_values` for tie breakers. Rank counts are useful for finding what kind of poker ranking hand it is, and ranks are used for the main hand components and kickers.

In [ ]:
def get_ranks_counts(cards: list[Card]) -> tuple[tuple[int, ...], tuple[int, ...]]:
    """Return ((ranks...), (counts...)) sorted by count then by rank value from cards"""
    items = Counter([rank_values[c.rank] for c in cards]).items()
    return tuple(zip(*sorted(items, key=lambda o: (o[1], o[0]), reverse=True)))

In [ ]:
get_ranks_counts(cards_fullhouse)

((3, 2, 10, 8), (3, 2, 1, 1))

In [ ]:
def test_get_ranks_counts():
    assert get_ranks_counts(cards_2kind) == ((2, 13, 11, 9, 7, 5), (2, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_3kind) == ((2, 11, 9, 7, 5), (3, 1, 1, 1, 1))
    assert get_ranks_counts(cards_4kind) == ((2, 12, 9, 6), (4, 1, 1, 1))
    assert get_ranks_counts(cards_2pair) == ((3, 2, 10, 8, 6), (2, 2, 1, 1, 1))
    assert get_ranks_counts(cards_fullhouse) == ((3, 2, 10, 8), (3, 2, 1, 1))
    assert get_ranks_counts(cards_highcard) == ((14, 12, 10, 8, 6, 4, 2), (1, 1, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_straight) == ((12, 10, 7, 6, 5, 4, 3), (1, 1, 1, 1, 1, 1, 1))

    # Edge case: two three-of-a-kinds (e.g. 222 333 K) — full house should pick higher triple
    cards_2trips = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]),
                    Card(suits[0], ranks[1]), Card(suits[1], ranks[1]), Card(suits[2], ranks[1]),
                    Card(suits[3], ranks[11])]
    assert get_ranks_counts(cards_2trips) == ((3, 2, 13), (3, 3, 1))

test_get_ranks_counts()

### flush

As long as the suit count is 5 or more, it is flush because we have seven cards total.

In [ ]:
def is_flush(cards: list[Card]) -> list:
    "If cards have flush, return the `rank_values` from the cards in sorted order. If not, return []"
    suit, count = Counter(c.suit for c in cards).most_common()[0]
    if count >= 5:
        return sorted([rank_values[c.rank] for c in cards if c.suit == suit][:5], reverse=True)
    return []

In [ ]:
def test_is_flush():
    # Flush: 5 spades, two side cards
    assert is_flush(cards_flush) == [10, 8, 6, 4, 2]

    # Straight flush
    assert is_flush(cards_straightflush) == [7, 6, 5, 4, 3]

    # Royal flush
    assert is_flush(cards_royalflush) == [14, 13, 12, 11, 10]

    # No flush: suits too spread out
    assert is_flush(cards_fullhouse) == []
    assert is_flush(cards_highcard) == []
    assert is_flush(cards_4kind) == []

    # Edge: all 7 cards same suit — returns top 5
    all_spades = [Card('♠', r) for r in 'A,K,Q,J,10,9,8'.split(',')]
    assert is_flush(all_spades) == [14, 13, 12, 11, 10]

    # Edge: exactly 5 same suit
    five_hearts = [Card('♥', r) for r in '2,3,4,5,6'.split(',')] + [Card('♠', 'A'), Card('♦', 'K')]
    assert is_flush(five_hearts) == [6, 5, 4, 3, 2]

test_is_flush()

### straight

Finding straight is trickier because Ace can serve as 1 or 14. Also, there are seven cards, but only five are used. 

In [ ]:
def is_straight(cards: list[Card]) -> int:
    "Return the highest value of the straight if cards contain 5 consecutive ranks, else 0"
    vals = sorted([rank_values[c.rank] for c in cards], reverse=True)
    if vals[0] == 14: 
        vals.append(1)
    for i in range(len(vals) - 4):
        if vals[i] - vals[i+4] == 4 and len(set(vals[i:i+5])) == 5:
            return vals[i]
    return 0

In [ ]:
def test_is_straight():
    # Regular straight: 3-4-5-6-7
    assert is_straight(cards_straight) == 7
    
    # Ace-high straight (10-J-Q-K-A)
    ace_high = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'),
                Card('♥','3'), Card('♦','6')]
    assert is_straight(ace_high) == 14
    
    # Wheel: A-2-3-4-5 (ace low)
    wheel = [Card('♠','A'), Card('♥','2'), Card('♦','3'), Card('♣','4'), Card('♠','5'),
             Card('♥','9'), Card('♦','J')]
    assert is_straight(wheel) == 5
    
    # No straight — gapped
    assert is_straight(cards_2pair) == 0
    
    # No straight — bunch of duplicates that run together in rank
    dupes = [Card('♠','5'), Card('♥','5'), Card('♦','6'), Card('♣','6'), Card('♠','7'),
             Card('♥','7'), Card('♦','8')]
    assert is_straight(dupes) == 0

test_is_straight()

### evaluate_hand

Now that we can detect flush and straight, we can update `evaluate_hand`.

In [ ]:
def evaluate_hand(cards: list[Card]) -> tuple:
    """Return hand rank tuple for the best 5-card hand from `cards`.
    Hands ranked from 1 (high card) to 10 (royal flush). Each rank tuple
    starts with the hand category, followed by tiebreaker values (kickers
    or relevant ranks).
    """
    ranks, counts = get_ranks_counts(cards)
    straight_val = is_straight(cards)
    flush_vals = is_flush(cards)
    if straight_val == 14: # royal flush
        return (10,)
    elif straight_val != 0 and flush_vals != []: # straight flush
        return (9, straight_val)
    elif counts[0] == 4: # four of a kind
        return (8, ranks[0], ranks[1])
    elif counts[0] == 3 and counts[1] >= 2: # full house
        return (7, ranks[0], ranks[1])
    elif flush_vals != []: # flush
        return (6, *flush_vals)
    elif straight_val != 0: # straight
        return (5, straight_val)
    elif counts[0] == 3: # three of a kind
        return (4, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2 and counts[1] == 2: # two pair
        return (3, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2: # one pair
        return (2, ranks[0], ranks[1], ranks[2], ranks[3])
    else: # high card
        return (1, *ranks)

In [ ]:
def test_evaluate_hand():
    # Royal flush: A♠ K♠ Q♠ J♠ 10♠ → (10,)
    assert evaluate_hand(cards_royalflush) == (10,)

    # Straight flush: 3♠ 4♠ 5♠ 6♠ 7♠ → (9, 7)
    assert evaluate_hand(cards_straightflush) == (9, 7)

    # Four of a kind: 2222 + Q 9 6 → (8, 2, 12)
    assert evaluate_hand(cards_4kind) == (8, 2, 12)

    # Full house: 333 22 + 10 8 → (7, 3, 2)
    assert evaluate_hand(cards_fullhouse) == (7, 3, 2)

    # Flush: 5 spades → (6, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_flush) == (6, 10, 8, 6, 4, 2)

    # Straight: 3-4-5-6-7 → (5, 7)
    assert evaluate_hand(cards_straight) == (5, 7)

    # Three of a kind: 222 + J 9 7 5 → (4, 2, 11, 9)
    assert evaluate_hand(cards_3kind) == (4, 2, 11, 9)

    # Two pair: 33 22 + 10 8 6 → (3, 3, 2, 10)
    assert evaluate_hand(cards_2pair) == (3, 3, 2, 10)

    # One pair: 22 + K J 9 7 5 → (2, 2, 13, 11, 9)
    assert evaluate_hand(cards_2kind) == (2, 2, 13, 11, 9)

    # High card: A Q 10 8 6 4 2 → (1, 14, 12, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_highcard) == (1, 14, 12, 10, 8, 6, 4, 2)

test_evaluate_hand()

### compare_hands

`compare_hands(hands: List[List[Card]]) -> tuple` — returns the ranks for the winner(s)

Winners are those who have the winning ranks

In [ ]:
def compare_hands(hands: list[list[Card]]) -> tuple:
    """Return the winning hand rank among `hands`."""
    return sorted([evaluate_hand(h) for h in hands], reverse=True)[0]

In [ ]:
evaluate_hand(cards_flush)

(6, 10, 8, 6, 4, 2)

In [ ]:
def test_compare_hands():
    # Clear winner: flush beats one pair
    assert compare_hands([cards_2kind, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Tie: two identical hands → both should win
    assert compare_hands([cards_flush, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Three hands, two tie for best
    assert compare_hands([cards_2kind, cards_flush, cards_royalflush]) == (10,)
    
    # Reverse order shouldn't matter
    assert compare_hands([cards_flush, cards_2kind]) == (6, 10, 8, 6, 4, 2)

test_compare_hands()

## Deck

In my original plan, I thought of writing a `CardCollection` class as an abstract class and create `Hand`, `Deck`, and `CommunityCards` by inheriting from it as they all consist of cards. However, they don't really have methods that share. To keep things simple, I decided to start implementing `Deck` as a list of `Card`s and see how it feels.

A deck consists of 52 cards. From that, we need to shuffle and withdraw cards so the dealer can deal cards.

In [ ]:
import random

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
deck[:5]

[♠2, ♠3, ♠4, ♠5, ♠6]

In [ ]:
def shuffle_deck(deck: list[Card]) -> list[Card]:
    "Shuffle `deck` in place and return it"
    random.shuffle(deck)
    return deck

In [ ]:
random.seed(42)

deck = [Card(s,r) for s in suits for r in ranks]
shuffle_deck(deck)[:5]

[♠J, ♥Q, ♥A, ♠5, ♥10]

In [ ]:
def withdraw_card(deck: list[Card], n: int=1) -> list[Card]:
    "Withdraw `n` cards from `deck` and return them. Modifies `deck` in place."
    return [deck.pop() for _ in range(n)]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
withdraw_card(deck)

[♣A]

In [ ]:
assert len(deck) == 51

In [ ]:
withdraw_card(deck, n=5)
assert len(deck) == 46

## Player

`Player`class:

- balance: int
- bet: int
- playing: bool
- hand: List[Card] methods:
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount).

In [ ]:
class Action(Enum):
    RAISE = auto()
    CALL = auto()
    CHECK = auto()
    FOLD = auto()

### Strategy

`Player` has a `strategy` attribute, which chooses actions.

In [ ]:
def human(player): 
    prompt = f"""This is {player.username}'s turn! Type an action: Call, Check, Fold, Raise X with X amount
     your current bet: {player.bet}, your hand: {player.hand}"""
    while True:
        act, *amnt = input(prompt).split()
        if (action := getattr(Action, act.upper(), None)):
            return (action, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
def always_call(player): return (Action.CALL, 0)

### player class

In [ ]:
@dataclass
class Player:
    username: str = 'new_user' # username
    balance: int = 0 # total balance
    bet: int = 0 # active bet this round
    playing: bool = True # playing this round or not
    hand: list = field(default_factory=list) # playing hand
    strategy: callable = always_call # a strategy the player uses to make actions (human or ai)

    def perform_action(self, stage_bet) -> tuple[Action, int, int]:
        ah = ActionHandler(self, stage_bet)
        return ah.handle()

In [ ]:
class ActionHandler:
    def __init__(self, player, stage_bet):
        self.player = player
        self.stage_bet = stage_bet
    
    def handle(self):
        action, amount = self.player.strategy(self.player)
        print(f'{self.player.username} intends to {action.name.lower()}, stage_bet: {self.stage_bet}, raise_amount: {amount}')
        return getattr(self, '_'+action.name.lower())(amount)

    def _call(self, amount):
        need = self.stage_bet - self.player.bet
        if self.player.balance < need:
            print(f"Cannot call. balance ({self.player.balance}) too low to match the stage_bet ({self.stage_bet}). Current bet: {self.player.bet}")
            return self.handle()
        else:
            self.player.balance -= need
            self.player.bet += need
            return need

    def _check(self, amount): return 0

    def _fold(self, amount):
        self.player.playing = False
        return 0

    def _raise(self, amount):
        """Player raises by amount, and update `stage_bet`"""
        new_stage_bet = self.stage_bet + amount
        need = new_stage_bet - self.player.bet
        if need > self.player.balance:
            print(f'Cannot raise by {amount}. Your balance: {self.player.balance}, current bet: {self.player.bet}, stage_bet: {self.stage_bet}')
            return self.handle()
        else:
            self.player.balance -= need
            self.player.bet += need
            return need

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_call)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to call, stage_bet: 2, raise_amount: 0


(2,
 Player(username='galopy', balance=18, bet=2, playing=True, hand=[], strategy=<function always_call at 0x79756f7062a0>))

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=human)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to raise, stage_bet: 2, raise_amount: 1


(3,
 Player(username='galopy', balance=17, bet=3, playing=True, hand=[], strategy=<function human at 0x79756f706700>))

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=human)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to fold, stage_bet: 2, raise_amount: 0


(0,
 Player(username='galopy', balance=20, bet=0, playing=False, hand=[], strategy=<function human at 0x79756f706700>))

## Round

Round Management

Attributes:
- stage (pre-flop, flop, turn, river)
- players
- deck
- community_cards

Methods:

- `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
- `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).
- `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.

In [ ]:
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

    def next(self):
        """Return the next stage, or None if already at RIVER."""
        return Stage(self.value + 1) if self.value < len(Stage) else None

Let's go through playing a round of poker game!

In [ ]:
class Round:
    def __init__(self, players = None, deck = None, community = None, stage = Stage.PREFLOP, stage_bet = 0, pot = 0):
        self.players = [] if players is None else players
        self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks]) if deck is None else deck
        self.community = [] if community is None else community
        self.stage = stage
        self.stage_bet = stage_bet
        self.pot = pot
    
    def state(self):
        "Return a string describing the current game situation."
        player = self.players[self._idx]
        return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | \
        Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

    def distribute(self, winners) -> None:
        "Distribute the prize pot to the winners"
        print(f"Distributing $ {self.pot} to {winners}")
        for player in winners:
            player.balance += self.pot // len(winners)

In [ ]:
@patch
def _prep_round(self: Round):
    "Prepare for the round by getting the deck, shuffle, deal hole hands."
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players: p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP

In [ ]:
@patch
def _reset_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.bet = 0

In [ ]:
@patch
def _deal_community(self: Round):
    "Deal community cards"
    if self.stage == Stage.FLOP: 
        self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): 
        self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")

In [ ]:
@patch
def _get_next_player(self: Round) -> Player:
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                act, new_stage_bet, bet_amount = yield player
            else:
                act, new_stage_bet, bet_amount = player.perform_action(self.stage_bet)
            self.pot += bet_amount
            if act == Action.RAISE:
                self._raiser = self._idx
                self.stage_bet = new_stage_bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break

In [ ]:
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self.distribute(winners)

In [ ]:
def play_action(action: str, round: Round, amount: int = 0):
    return getattr(player, '_' + action.name.lower())(round.stage_bet, amount)

In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._deal_community()
        self._reset_betting_round()
        yield from self._get_next_player()
    self._showdown()

I am thinking of pulling the action logics such as `_call` out of `Player` to simplify the class. Maybe I can put them into `Round`? Or a separate class and inherit from it? I also want to simplify `Round` class as well. I think it is getting quite big. But I am not sure if it is too big of a class with much complexity or it is ok. What is your thought?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great questions — both about separation of concerns and class size.

**On moving actions out of `Player`:** The `_call`, `_check`, `_fold`, `_raise` methods tightly couple `Player` and `stage_bet` — they modify player state (balance, bet, playing) but also need round state (stage_bet) to function. That's why you're feeling the friction. Moving them *into* `Round` would solve the coupling but make `Round` even bigger, which works against your other goal.

A third option: a dedicated class (say `BettingRound` or `ActionHandler`) that holds a reference to the player and the current `stage_bet`. That way `Player` stays a simple data container, `Round` stays focused on orchestration, and the betting logic lives in its own place. You'd be following the **single responsibility principle** — each class has one job.

**On `Round` being too big:** It has about 7 methods, which isn't huge by itself. The real complexity is in `_get_next_player` with the generator-based flow control — that's where the logic gets tricky. If you extract the betting logic, `Round` naturally shrinks and becomes mostly orchestration (prep, deal, advance stages, showdown).

What feels like the most natural split to you? Do you lean toward the separate class approach, or does something else feel right?

<details class='token-usage-details' markdown='1'><summary>$0.0649</summary>

`total=29,257 | in=28,492 | out=765 | cached=0.0% | $0.0649 | glm-5p2`

</details>


I like the third option. Tell me more about it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the core idea: a class that encapsulates *one betting action* — it knows the player, the current `stage_bet`, and how to execute each action type.

The responsibilities would split like this:

- **`Player`** — pure data: username, balance, bet, hand, playing, strategy. No action logic at all.
- **`ActionHandler`** (or whatever name you like) — takes a player and `stage_bet`, executes the action (`call`, `check`, `fold`, `raise`), and returns how much was added to the pot. It modifies player state but doesn't know about `Round`.
- **`Round`** — orchestrates the game flow: stages, dealing, betting rounds, showdown. It creates an `ActionHandler` when a player needs to act, and uses the result.

The key benefit is that `Round` no longer needs to know *how* a call or raise works internally — it just says "this player is acting, handle it" and gets back the pot contribution. And `Player` becomes a clean data container that's easy to test and reason about.

One design question to think about: should `ActionHandler` be a short-lived object (created fresh each time a player acts) or a longer-lived helper that `Round` holds a reference to? What feels more natural to you?

<details class='token-usage-details' markdown='1'><summary>$0.0092</summary>

`total=29,145 | in=28,847 | out=298 | cached=97.8% | $0.0092 | glm-5p2`

</details>


So `ActionHandler` also has a logic to verify player actions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, exactly. The verification logic — like checking if a player has enough balance to call, or whether checking is even allowed (stage_bet must equal player.bet) — fits naturally in `ActionHandler` since it already has both the player state and `stage_bet` it needs to make those checks.

So `ActionHandler` would do two things:
1. **Validate** — is this action legal given the current state?
2. **Execute** — update player state and return the pot contribution

That keeps all the "rules of betting" in one place, separate from both the player data and the round orchestration.

One thing to consider: what should happen when an action is invalid? Right now your code recursively calls `perform_action` again. With `ActionHandler`, you might return a result indicating success or failure, and let `Round` decide what to do — like re-prompting the player. That gives `Round` control over the flow without `ActionHandler` needing to know about re-prompting.

What do you think — should `ActionHandler` handle retries internally, or just report back and let `Round` decide?

<details class='token-usage-details' markdown='1'><summary>$0.0093</summary>

`total=29,521 | in=29,196 | out=325 | cached=98.1% | $0.0093 | glm-5p2`

</details>


I want my classes to be very simple and lean as possible. I think letting `Round` take care of failed actions is bad considering we have `ActionHandler`. It should handle actions.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That makes sense — if `ActionHandler` owns actions, it should own the full lifecycle including retries. `Round` just calls it and gets back a clean result: the pot contribution. `Round` doesn't even need to know that a retry happened.

So the split becomes:

- **`ActionHandler`** — validate, execute, retry on failure. Returns pot contribution.
- **`Round`** — "player needs to act" → calls `ActionHandler` → gets pot amount → moves on.

`Round` never sees the messy details. One thing to think about: since `ActionHandler` needs to re-prompt on failure, it'll need access to the player's `strategy` to ask for a new action. That's fine since it already has the player. Just something to keep in mind as you sketch it out.

Ready to start writing `ActionHandler`, or still thinking through the design?

<details class='token-usage-details' markdown='1'><summary>$0.0088</summary>

`total=29,765 | in=29,524 | out=241 | cached=98.2% | $0.0088 | glm-5p2`

</details>


You mentioned about `ActionHandler` being short-lived or longer-lived. What difference does it make?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The difference is about **what state it holds and when**:

**Short-lived** — created fresh each time a player acts, then discarded. It only knows about the current player and current `stage_bet`. After the action resolves, it's gone. Simple, no lingering state, easy to reason about. But you're creating a new object every action.

**Longer-lived** — `Round` holds one instance that persists across the betting round. It could track things like "how many times has this player tried to raise and failed" or other per-round betting state. But it needs to be reset between betting rounds, and it accumulates state that might not be necessary.

For your goal of keeping classes lean and simple, **short-lived** seems like the better fit — no state to manage between actions, no reset logic, and it does one thing then goes away. The object creation overhead is negligible for a game like this.

Does that align with what you're thinking?

<details class='token-usage-details' markdown='1'><summary>$0.0087</summary>

`total=30,013 | in=29,787 | out=226 | cached=98.4% | $0.0087 | glm-5p2`

</details>


yeah

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Then you're in good shape design-wise. When you're ready, you can start by sketching out `ActionHandler` — its `__init__` (what it takes), and the action methods you'd move over from `Player`. The existing `_call`, `_check`, `_fold`, `_raise` logic is basically ready to move; it just needs a new home.

What would you like to tackle first?

<details class='token-usage-details' markdown='1'><summary>$0.0080</summary>

`total=30,168 | in=30,044 | out=124 | cached=98.5% | $0.0080 | glm-5p2`

</details>


I want to try to use a `dataclass` for `Player` as it only contains variables to keep track of. Is this a good idea? How is it different from regular class and namedtuple? What are pros and cons?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great question. All three options work, but they have different tradeoffs for your case:

**`namedtuple`** (what you used for `Card`)
- Immutable, lightweight, no custom logic
- Fields are fixed at creation — can't add/modify attributes after
- Good for simple value objects like `Card`
- But `Player` needs mutable state (balance, bet, playing all change), so namedtuple is a poor fit

**Regular class** (what you have now)
- Full flexibility — methods, properties, inheritance
- But you don't need much behavior anymore if actions move to `ActionHandler`
- More boilerplate (`__init__`, `__repr__`)

**`dataclass`**
- Auto-generates `__init__`, `__repr__`, `__eq__` for you — less boilerplate
- Mutable by default (perfect for `Player` — balance, bet, playing change constantly)
- Can still add methods if needed (e.g., `perform_action` if you keep it, or simple helpers)
- Can make individual fields immutable with `frozen=True`, but you'd want it mutable
- Supports default values cleanly

For `Player` as a lean data container, `dataclass` seems like the sweet spot — you get clean syntax, auto-generated boilerplate, and mutability. The main con is it's slightly less flexible than a regular class for complex inheritance, but you're not doing that.

Want to try converting `Player` to a `dataclass`?

<details class='token-usage-details' markdown='1'><summary>$0.0087</summary>

`total=30,577 | in=30,232 | out=345 | cached=99.8% | $0.0087 | glm-5p2`

</details>


In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_call)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to call, stage_bet: 2, raise_amount: 0


(2,
 Player(username='galopy', balance=18, bet=2, playing=True, hand=[], strategy=<function always_call at 0x79757a95a200>))

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=human)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to raise, stage_bet: 2, raise_amount: 30


Cannot raise by 30. Your balance: 20, current bet: 0, stage_bet: 2


galopy intends to raise, stage_bet: 2, raise_amount: 1


(3,
 Player(username='galopy', balance=17, bet=3, playing=True, hand=[], strategy=<function human at 0x79757a95a2a0>))

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=human)
need = player1.perform_action(stage_bet)
need, player1

galopy folded, stage_bet: 2, raise_amount: 0


(0,
 Player(username='galopy', balance=20, bet=0, playing=False, hand=[], strategy=<function human at 0x79757a95a2a0>))

In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=human)
player2 = Player('jalopy', balance=20, playing=True)
players = [player1, player2]
round1 = Round(players = players)
game = round1.start_round()
player = next(game)
player


--- PREFLOP --- Community: []


Player username: galopy, balance: 20, bet: 0,     hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
def act(game, player, round):
    try:
        player = game.send(player.perform_action(round.stage_bet))
    except StopIteration:
        player = None
    return player

In [ ]:
player = act(game, player, round1)
player

action: Action.CHECK, stage_bet: 0, raise_amount: 0


galopy checked


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 0, raise_amount: 0


jalopy called by betting 0. Total self.bet: 0



--- FLOP --- Community: [♥Q, ♦K, ♦J]


Player username: galopy, balance: 20, bet: 0,     hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: FLOP | Hand: [♣7, ♠J] |         Community: [♥Q, ♦K, ♦J] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"

In [ ]:
player = act(game, player, round1)
player

action: Action.RAISE, stage_bet: 0, raise_amount: 2


galopy raised by 2 to stage_bet: 2!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 2, raise_amount: 0


jalopy called by betting 2. Total self.bet: 2



--- TURN --- Community: [♥Q, ♦K, ♦J, ♣K]


Player username: galopy, balance: 18, bet: 0,     hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: TURN | Hand: [♣7, ♠J] |         Community: [♥Q, ♦K, ♦J, ♣K] | Stage bet: 0 | Your bet: 0 | Balance: 18 | Pot: 4"

In [ ]:
player = act(game, player, round1)
player

action: Action.RAISE, stage_bet: 0, raise_amount: 3


galopy raised by 3 to stage_bet: 3!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 3, raise_amount: 0


jalopy called by betting 3. Total self.bet: 3



--- RIVER --- Community: [♥Q, ♦K, ♦J, ♣K, ♦3]


Player username: galopy, balance: 15, bet: 0,     hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: RIVER | Hand: [♣7, ♠J] |         Community: [♥Q, ♦K, ♦J, ♣K, ♦3] | Stage bet: 0 | Your bet: 0 | Balance: 15 | Pot: 10"

In [ ]:
player = act(game, player, round1)
player

action: Action.RAISE, stage_bet: 0, raise_amount: 4


galopy raised by 4 to stage_bet: 4!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 4, raise_amount: 0


jalopy called by betting 4. Total self.bet: 4


Winners: ['galopy'] with (3, 13, 11, 12)


Distributing $ 18 to [Player username: galopy, balance: 11, bet: 4,     hand: [♣7, ♠J], playing: True, human: Y]


I think I have `Player` and `Round` working now. How does it look?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You've made great progress — the core flow from dealing through showdown actually runs end to end. A few observations:

**What's working well**

- `Card` as a `namedtuple` with patched `__repr__`, `__lt__`, and `__eq__` is clean and practical.
- `evaluate_hand` is thorough and well-tested — the two-triples edge case in full house detection is a nice catch.
- The generator-based approach in `_get_next_player` / `start_round` is clever. Yielding only on human turns and letting AI players resolve internally keeps the interactive loop simple.

You'll want to swap to single quotes or use a variable.

**Betting round logic**

The loop termination in `_get_next_player` is subtle but looks sound from my trace-through. One edge case worth thinking about: what happens when all but one player folds mid-round? The round should end immediately without waiting for `_idx` to wrap around to `_raiser` — it might work already depending on player order, but it's worth walking through with a pen-and-paper example.

Also, your TODO list still has blinds and dealer rotation, which would affect where `_reset_betting_round` starts. That's the natural next area to tackle when you're ready.

What aspect do you want to dig into next?

<details class='token-usage-details' markdown='1'><summary>$0.0146</summary>

`total=36,759 | in=31,600 | out=5,159 | cached=68.9% | reasoning=4,815 | $0.0146 | deepseek-v4-pro`

</details>


Ok, so I want to write tests for `Player` and `Round` classes before I move on. How can I get started?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's start with `Player` since it's simpler and has the clearest contracts. Think about each method's behavior in different scenarios.

For `_call`, for example, you'd want to cover:

- When the player has enough balance to match `stage_bet`
- When they don't have enough
- When `self.bet` already equals `stage_bet` (should cost 0)

For `_raise`, similarly: sufficient balance, insufficient balance, and the resulting `stage_bet` change.

One design question worth settling first: what should happen when a player can't afford a call or raise? Right now it calls `perform_action` again to retry, which could loop infinitely with an AI strategy. Would you want to auto-fold instead, or handle it differently?

Pick one method — say `_call` — and sketch a test or two. I'll look over what you come up with.

<details class='token-usage-details' markdown='1'><summary>$0.0040</summary>

`total=32,477 | in=31,965 | out=512 | cached=93.3% | reasoning=321 | $0.0040 | deepseek-v4-pro`

</details>


Sounds good. But, I would like to see if I can simplify the `Player` class further. I think the arguments for the methods such as `stage_bet` and `*args` may be unnecessary. 

Let's think about what happens in actions. There are four actions (check, call, fold, raise). When players try to make an action, they can see the `stage_bet` through the `Round` object. Comparing it with the `Player.bet` and `Player.balance`, we can figure out what actions players can take. Players can check only if `stage_bet == Player.bet`. They can always fold. They can call if `stage_bet != Player.bet and balance > need`. They can raise if balance is enough.

I also think I can simplify return values from those action methods. We can only return their new bet amount. By comparing the current stage_bet with players' new bets, we can figure out the new stage_bet. And we can figure out what actions they took from this number as well.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

This is a really good simplification. You're essentially saying: instead of having each action method compute and return *what changed*, just have it update the player's own state (`self.bet`, `self.playing`, `self.balance`) and let the `Round` inspect the result afterward. That's a cleaner separation of concerns.

Let's walk through what `_get_next_player` would look like under this model:

- Before the action, the round notes the current `stage_bet` and the player's current `bet`
- It calls `player.perform_action(self)` — passing the round so the player can read `round.stage_bet`
- Afterward, it compares `player.bet` to the old `stage_bet`: if higher, it's a raise; if equal (and player is still playing), it's a call or check; if `player.playing` is `False`, it's a fold
- The pot update is just the difference in the player's bet

A few edge cases worth thinking through:

- When a player raises, the new `stage_bet` becomes that player's new bet. Everyone else (including players who already acted this round) now needs to match it. Does your current `_raiser` / `_idx` loop handle that correctly with this new approach?
- When a player *can't* afford a call, what should happen? Auto-fold? Or should the player's strategy still get a say? The behavior changes depending on how you answer that.

The fold detection through `player.playing` is also nice — it means the round can check after *every* action whether only one player remains, and skip straight to showdown.

Want to sketch what the simplified `Player` class would look like?

<details class='token-usage-details' markdown='1'><summary>$0.0197</summary>

`total=32,041 | in=30,901 | out=1,140 | cached=10.8% | reasoning=776 | $0.0197 | deepseek-v4-pro`

</details>


In [ ]:
def human(player): 
    prompt = f"""This is {player.username}'s turn! Type an action: Call, Check, Fold, Raise X with X amount
     your current bet: {player.bet}, your hand: {player.hand}"""
    while True:
        act, *amnt = input(prompt).split()
        if (action := getattr(Action, act.upper(), None)):
            return (action, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
def always_call(player): return (Action.CALL, 0)

In [ ]:
class Player:
    def __init__(self, username, balance = 0, bet = 0, playing = True, hand = None, strategy = always_call):
        self.username = username                    # username
        self.balance = balance                      # total balance
        self.bet = bet                              # active bet this round
        self.playing = playing                      # playing this round or not
        self.hand = [] if hand is None else hand    # playing hand
        self.strategy = strategy                    # a strategy the player uses to make actions (human or ai)
    
    def perform_action(self, stage_bet) -> tuple[Action, int, int]:
        """Prompt player for an action, then call the appropriate function.
        To keep playing the round, the player has to match the `stage_bet`
        It returns a tuple of (`Action`, updated stage_bet, and player's bet)."""
        action, amount = self.strategy()
        print(f'{self.username} {action.name.lower()}ed, stage_bet: {stage_bet}, raise_amount: {amount}')
        return getattr(self, '_'+action.name.lower())(stage_bet, amount)

    def _call(self, stage_bet, amount):
        need = stage_bet - self.bet
        if self.balance < need:
            print(f"Cannot call. balance ({self.balance}) too low to match the stage_bet ({stage_bet}). Current bet: {self.bet}")
            return self.perform_action(stage_bet)
        else:
            self.balance -= need
            self.bet += need
            return need

    def _check(self, stage_bet, amount): return 0

    def _fold(self, stage_bet, amount):
        self.playing = False
        return 0

    def _raise(self, stage_bet, amount):
        """Player raises by amount, and update `stage_bet`"""
        new_stage_bet = stage_bet + amount
        need = new_stage_bet - self.bet
        if need > self.balance:
            print(f'Cannot raise by {amount}. Your balance: {self.balance}, current bet: {self.bet}, stage_bet: {stage_bet}')
            return self.perform_action(stage_bet)
        else:
            self.balance -= need
            self.bet += need
            return need

    def __repr__(self): return f"Player username: {self.username}, balance: {self.balance}, bet: {self.bet}, \
    hand: {self.hand}, playing: {self.playing}, human: {"Y" if self.strategy == human else "N"}"

I simplified a bit. Should I pass `round` to `player.perform_action`? I only need the `stage_bet` from the object. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Passing just `stage_bet` is the right call — it keeps `Player` decoupled from `Round` and you only need that one value for the action logic. If strategies later need more context (community cards, pot, number of active players), you can revisit then, but for now simpler is better.

One thing I noticed: your strategies (`human`, `always_call`) take `player` as an argument, but `perform_action` calls `self.strategy()` with no arguments. Want to fix that next, or are you still thinking through the design?

<details class='token-usage-details' markdown='1'><summary>$0.0038</summary>

`total=34,036 | in=33,263 | out=773 | cached=96.6% | reasoning=655 | $0.0038 | deepseek-v4-pro`

</details>


In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_call)
act, stage_bet, bet = player1.perform_action(stage_bet)

TypeError: always_call() missing 1 required positional argument: 'player'

Isn't `self` passed in as the first argument? I thought it expands into `strategy(player)`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It's a subtle point. `self.strategy` is a plain function stored as an attribute — it's not a method bound to a class. So calling `self.strategy()` doesn't automatically pass `self` the way `self.perform_action()` does. You need to pass it explicitly: `self.strategy(self)`.

Same idea as if you stored `len` as an attribute — you'd still need to call `self.func(x)`, not `self.func()`.

<details class='token-usage-details' markdown='1'><summary>$0.0031</summary>

`total=34,049 | in=33,806 | out=243 | cached=96.9% | reasoning=141 | $0.0031 | deepseek-v4-pro`

</details>


`self.strategy(self)` seems redundant tho

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It *looks* redundant, but the two `self`s serve different roles. The first is attribute access — reaching into the instance to grab whatever function is stored as `strategy`. The second is passing the player object *into* that function so the strategy can inspect the hand, balance, etc.

Think of it like this: if you stored a function in a dict, you'd write `d['func'](arg)`. Here it's `self.strategy(self)` — same pattern, just the instance is both the container and the argument.

If it bothers you, you could flip it: have `perform_action` pass `self` to a standalone dispatch function that looks up the strategy and calls it. But that adds indirection for not much gain. The explicit `self.strategy(self)` is clear enough once you get used to it.

<details class='token-usage-details' markdown='1'><summary>$0.0037</summary>

`total=34,669 | in=33,971 | out=698 | cached=97.2% | reasoning=522 | $0.0037 | deepseek-v4-pro`

</details>


In [ ]:
stage_bet, bet, player1

NameError: name 'bet' is not defined

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=human)
act, stage_bet, bet = player1.perform_action(stage_bet)

TypeError: human() missing 1 required positional argument: 'player'

In [ ]:
stage_bet, bet, player1

NameError: name 'bet' is not defined

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

### Some notes at the end to consider:

I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great question — it's worth stepping back and thinking about the design before committing.

The current approach works, but the `raiser, idx = -1, 0` with the `if raiser == -1: raiser = 0` hack is a bit fragile. The core challenge is: **after someone raises, everyone else needs to act again.** How you model that determines the cleanliness.

Here are three approaches worth considering:

**1. Queue/deque approach** — Probably the cleanest conceptually. Keep a `deque` of players who still need to act. Pop the front player, they act. If they call or check, they're done (don't re-add). If they raise, re-add all other *active* players to the queue. When the queue is empty, the round is over. No index tracking, no `raiser` variable.

**2. Flag-based approach** — Track a `has_acted` flag per player. After a raise, reset everyone's flag. The round ends when all active players have `has_acted = True`. Simple state, explicit termination condition.

**3. Generator/cycle approach** — Use `itertools.cycle` over active players, yielding each in turn. Break out when all active bets match `stage_bet` and everyone has had a chance to act since the last raise. This is more "functional" but the termination logic can get tricky.

Norvig tends to favor the simplest thing that clearly works. The deque approach maps very naturally to the problem — "who still needs to act?" is literally the queue contents. It also avoids the circular indexing math entirely.

Which of these resonates with you? Or would you like to explore one in more detail?

<details class='token-usage-details' markdown='1'><summary>$0.0892</summary>

`total=42,850 | in=41,930 | out=920 | cached=6.2% | $0.0892 | glm-5p2`

</details>


I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good question to step back and reflect on. Let's look at a few approaches.

Your current approach uses a `while idx != raiser` loop with mutable state. It works, but you're right to ask whether there's a cleaner way.

Here are three alternative approaches for the betting round:

---

**1. Generator approach**

A generator yields control back to the caller after each action, letting an outer loop drive the game:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    raiser = -1
    idx = 0
    while idx != raiser:
        if raiser == -1: raiser = 0
        player = self.players[idx]
        if not player.playing:
            idx = (idx + 1) % len(self.players)
            continue
        yield player  # <-- caller gets the player, performs action, updates state
        idx = (idx + 1) % len(self.players)
```

Caller:
```python
for player in round.betting_round():
    act, new_stage_bet, bet = player.perform_action(round.stage_bet)
    round.pot += bet
    if act == Action.RAISE:
        round.stage_bet = new_stage_bet
        # need to communicate raiser index back somehow — this is the tricky bit
```

The generator is elegant in theory, but communicating the raiser index *back* into the generator is awkward. Norvig *did* use generators in cs212, but usually when the flow is one-directional.

---

**2. Explicit action queue (event-driven)**

Each action produces the next state explicitly:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    actor_idx = self._next_active(0)  # first active player
    last_raiser = actor_idx
    
    while True:
        player = self.players[actor_idx]
        act, new_stage_bet, bet = player.perform_action(self.stage_bet)
        self.pot += bet
        
        if act == Action.RAISE:
            last_raiser = actor_idx
            self.stage_bet = new_stage_bet
        elif act == Action.FOLD:
            player.playing = False
        
        actor_idx = self._next_active((actor_idx + 1) % len(self.players))
        if actor_idx == last_raiser: break  # everyone acted
```

This separates the state update from the iteration logic, which is clearer.

---

**3. Norvig-style: separate "find actor" and "all acted?"**

Norvig tends to extract helper functions so the main loop reads cleanly:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    last_raiser = self.first_active()
    
    while True:
        player = self.next_player()
        yield self._handle(player)
        if self.done(last_raiser): break
```

Each helper does one thing, and the main loop is almost plain English.

---

Which of these directions feels most like what you want? Or do you want to sketch one out together before picking?

<details class='token-usage-details' markdown='1'><summary>$0.0290</summary>

`total=46,470 | in=45,452 | out=1,018 | cached=7.3% | reasoning=305 | $0.0290 | deepseek-v4-pro`

</details>


I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a good instinct. Norvig tends to favor explicitness and simplicity over cleverness — the generator is a bit "magical" and harder to debug.

Here's a simpler approach: **manual stepping with two methods.**

- `round.state()` — returns a string describing whose turn it is and the game situation (no action taken, just information)
- `round.play_action(action, amount)` — takes your decision and advances one player's turn (AI players auto-advance through themselves until the next human)

You'd use it like:

```python
# Cell 1: see what's happening
round.state()  # "galopy's turn (PREFLOP). Hand: ♠A ♠K..."

# Cell 2: act
round.play_action(Action.RAISE, 5)

# Cell 3: see next state (AI players already processed)
round.state()  # "galopy's turn (PREFLOP). Opponent raised to 10..."

# Cell 4: act
round.play_action(Action.CALL, 0)
```

Each cell does exactly one thing, `round` holds all the game state visibly, and there's no hidden loop or yield mechanism. You can inspect `round` at any point between cells — check `round.players`, `round.community`, or even run `evaluate_hand` on your cards.

The downside is you need to manage the stage transitions yourself (or `play_action` handles them automatically when a betting round ends). But that's a small bit of logic to add.

How does this feel compared to the generator approach?

<details class='token-usage-details' markdown='1'><summary>$0.0076</summary>

`total=50,919 | in=50,206 | out=713 | cached=88.5% | reasoning=379 | $0.0076 | deepseek-v4-pro`

</details>


What would be nice approach if I eventually want to expand it to more players? And those players all have their actions on their separate computers. And everyone waits until it is their turn.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The manual stepping approach actually maps beautifully to a multi-player setup. Think of it as a **server** holding the `Round` state, with each player connecting from their own computer. The server's job is simple:

1. Figure out whose turn it is
2. Send that player the game state
3. Wait for their action
4. Apply it, repeat

Two clean approaches:

**A web server** (like FastAPI) — each player hits an endpoint. The server runs `round.play_action(...)` when it receives a decision, then pushes the updated state to whoever's turn is next. solveit could even act as the server, since you already have a running Python kernel:

```
Player A (browser)  →  solveit kernel  →  Player B (browser)
         "your turn: ♠A ♠K..."         "opponent raised to 10..."
```

**A simpler option for now**: keep everything in one solveit dialog and use the manual stepping pattern. When you're ready for multiplayer, extract `Round` into a proper server. The key point is that `round.state()` and `round.play_action(...)` are already a clean API — they don't care whether the caller is a local cell or a remote HTTP request.

So the manual stepping approach isn't just simple for now — it's also the right foundation for multiplayer later. Want to move forward with that?

<details class='token-usage-details' markdown='1'><summary>$0.0046</summary>

`total=51,150 | in=50,633 | out=517 | cached=97.8% | reasoning=225 | $0.0046 | deepseek-v4-pro`

</details>
